# Tutorial: register_dataset

In [ ]:
import json
import os
import tempfile

import nibabel as nib
import numpy as np

# Create a temporary base directory for the entire tutorial
tmpdir = tempfile.mkdtemp(prefix="asparagus_tutorial_")
data_root = os.path.join(tmpdir, "asparagus_data")
os.makedirs(data_root, exist_ok=True)

# ASPARAGUS_DATA must be set before importing asparagus_preprocessing
os.environ["ASPARAGUS_DATA"] = data_root

from asparagus_preprocessing.utils.dataset_registration import register_dataset

print(f"Temporary directory: {tmpdir}")
print(f"ASPARAGUS_DATA:      {data_root}")

# Create synthetic NIfTI files

In [ ]:
nifti_dir = os.path.join(tmpdir, "raw_niftis")
os.makedirs(nifti_dir, exist_ok=True)

n_files = 20
shape = (16, 16, 16)  # small volumes to keep it fast
affine = np.eye(4)

for i in range(n_files):
    data = np.random.randn(*shape).astype(np.float32)
    img = nib.Nifti1Image(data, affine)
    nib.save(img, os.path.join(nifti_dir, f"sub_{i:03d}.nii.gz"))

print(f"Created {n_files} NIfTI files in {nifti_dir}")
print(os.listdir(nifti_dir)[:5], "...")

# Directory mode

In [ ]:
output_dir_auto = register_dataset(
    task_name="PT001_Tutorial",
    n_classes=1,
    n_modalities=1,
    data_dir=nifti_dir,
    extension=".nii.gz",
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    n_folds=5,
    seed=42,
)

# Inspect output files

In [ ]:
print("Generated files:")
for f in sorted(os.listdir(output_dir_auto)):
    print(f"  {f}")

print("\n--- dataset.json ---")
with open(os.path.join(output_dir_auto, "dataset.json")) as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
print("--- paths.json (first 5 entries) ---")
with open(os.path.join(output_dir_auto, "paths.json")) as f:
    paths = json.load(f)
print(f"Total paths: {len(paths)}")
for p in paths[:5]:
    print(f"  {p}")

In [ ]:
print("--- split_80_10_10.json (fold 0) ---")
with open(os.path.join(output_dir_auto, "split_80_10_10.json")) as f:
    splits = json.load(f)
print(f"Number of folds: {len(splits)}")
print(f"Fold 0 train: {len(splits[0]['train'])} files")
print(f"Fold 0 val:   {len(splits[0]['val'])} files")

print("\n--- TEST_80_10_10.json ---")
with open(os.path.join(output_dir_auto, "TEST_80_10_10.json")) as f:
    test_files = json.load(f)
print(f"Test files: {len(test_files)}")
for p in test_files:
    print(f"  {p}")

# Explicit mode

In [ ]:
all_files = sorted(
    os.path.join(nifti_dir, f) for f in os.listdir(nifti_dir) if f.endswith(".nii.gz")
)

train_files = all_files[:14]
val_files = all_files[14:17]
test_files = all_files[17:]

output_dir_explicit = register_dataset(
    task_name="PT002_TutorialExplicit",
    n_classes=1,
    n_modalities=1,
    train=train_files,
    val=val_files,
    test=test_files,
    n_folds=3,
)

In [ ]:
print("Generated files:")
for f in sorted(os.listdir(output_dir_explicit)):
    print(f"  {f}")

print("\n--- dataset.json ---")
with open(os.path.join(output_dir_explicit, "dataset.json")) as f:
    print(json.dumps(json.load(f), indent=2))

print("\n--- split_manual.json (fold 0) ---")
with open(os.path.join(output_dir_explicit, "split_manual.json")) as f:
    splits = json.load(f)
print(f"Number of folds: {len(splits)}")
print(f"Fold 0 train: {len(splits[0]['train'])} files")
print(f"Fold 0 val:   {len(splits[0]['val'])} files")

# Cleanup

In [ ]:
import shutil

shutil.rmtree(tmpdir)
del os.environ["ASPARAGUS_DATA"]

print("Cleaned up temporary files and unset ASPARAGUS_DATA")